# Notebook for cleaning ISO country code dataset from kaggle. Needed to make a dim table for silver_spotify_daily table to get out country instead of just ISO alpha-2 code.

In [4]:
import pandas as pd
import duckdb

FILE_PATH = "../data/raw/country_codes.csv"
PARQUET_PATH = "../data/processed/spotify_daily_top_songs_clean_version608.parquet"

df = pd.read_csv(FILE_PATH)
dfp = pd.read_parquet(PARQUET_PATH)

In [12]:
# LOL - Poor Namibia...
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 249 entries, 0 to 248
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   country  249 non-null    str  
 1   alpha2   248 non-null    str  
 2   alpha3   249 non-null    str  
 3   numeric  249 non-null    int64
dtypes: int64(1), str(3)
memory usage: 12.1 KB


In [11]:
dfp['country'].value_counts()

country
DO    29176
IT    29174
NI    29169
PL    29164
HU    29163
      ...  
VN    28406
UY    28402
LU    28315
VE    28262
GB    20919
Name: count, Length: 73, dtype: int64

In [13]:
# Gör en kopia av de två columns jag behöver.
df_dim = df[['alpha2', 'country']].copy()

In [14]:
# DÖP OM DOM!
# Om båda tabellerna har en column som heter country kommer det bli extremt förvirrande när vi gör queries.
df_dim = df_dim.rename(columns={'alpha2': 'iso_code', 'country': 'country_name'})

In [15]:
# Fixa NAMIBIA "buggen" (NA == NaN i pandas rofl)
df_dim['iso_code'] = df_dim['iso_code'].fillna('NA')

In [ ]:
# VALIDERINGS CHECK. Matchar mina dataset?
# Hämta alla unika länder ifrån spotify datan
spotify_countries = dfp['country'].unique()

# Kolla om några av mina Spotify koder saknas i nya dimensionstabellen
missing_in_dim = [c for c in spotify_countries if c not in df_dim['iso_code'].values]

print(f"Spotify länder som SAKNAR matchning i min dim-tabell: {missing_in_dim}")

Spotify länder som SAKNAR matchning i min dim-tabell: ['Global']


In [19]:
# Allting stämmer. Inga länder saknar matchning, global är inget land och finns inte i ISO standarden.
# Spara min dim table.
dim_path = "../data/processed/dim_geography.parquet"
df_dim.to_parquet(dim_path, index=False)

print(f"Dimensiontabell sparad: {dim_path}")

Dimensiontabell sparad: ../data/processed/dim_geography.parquet
